In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torchvision import datasets,transforms,models
from torch.utils.data import DataLoader
import os
import numpy as np
from torch.utils.data import Dataset
from PIL import Image, ImageOps
import cv2
from torch.utils.data import random_split

!pip install onnxruntime
!pip install rembg
from rembg import remove

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.7/54.7 kB 5.5 MB/s eta 0:00:00


In [ ]:
%cd /content/drive/MyDrive

/content/drive/MyDrive


In [ ]:

# ---------------------------------------------------------------
# Fix the orientation so the image is "north-aligned"
# ---------------------------------------------------------------
def correct_orientation(img):
    try:
        exif = img._getexif()
        if exif is not None:
            for tag, val in ExifTags.TAGS.items():
                if val == 'Orientation':
                    orientation_tag = tag
                    break

            orientation = exif.get(orientation_tag, 1)

            if orientation == 3:
                img = img.rotate(180, expand=True)
            elif orientation == 6:
                img = img.rotate(270, expand=True)
            elif orientation == 8:
                img = img.rotate(90, expand=True)
    except Exception:
        pass

    return img


# ---------------------------------------------------------------
# Per-image Z-Normalization: (x - mean) / std
# ---------------------------------------------------------------
class ZNormalize(object):
    def __call__(self, tensor):
        mean = tensor.mean()
        std = tensor.std()

        # Avoid divide-by-zero
        if std < 1e-6:
            std = 1e-6

        return (tensor - mean) / std


# ---------------------------------------------------------------
# Custom dataset with background removal + orientation correction
# ---------------------------------------------------------------
class PreprocessedImageDataset(Dataset):
    def __init__(self, dataset, img_size=512):
        self.dataset = dataset

        # Post-background transforms
        self.post_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            ZNormalize(),     # <-- PER-IMAGE Z-NORM HERE
        ])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        path, label = self.dataset.dataset.samples[self.dataset.indices[idx]]

        # Load image
        img = Image.open(path).convert("RGB")

        # Step 1 — rotate to north
        img = correct_orientation(img)

        # Step 2 — remove background
        img = remove(img)

        # Step 3 — resize, tensor, z-normalize
        img = self.post_transform(img)

        return img, label


# ---------------------------------------------------------------
# LOAD ORIGINAL DATASET
# ---------------------------------------------------------------
dataset = datasets.ImageFolder("Image_Dataset")

# 80/20 split
train_len = int(0.8 * len(dataset))
test_len = len(dataset) - train_len

train_base, test_base = random_split(dataset, [train_len, test_len])

# Wrap both subsets with preprocessing
train_ds = PreprocessedImageDataset(train_base, img_size=128)
test_ds = PreprocessedImageDataset(test_base, img_size=128)

# Data loaders
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

print("Train samples:", len(train_ds))
print("Test samples:", len(test_ds))


Train samples: 21292
Test samples: 5324


In [ ]:
model = torchvision.models.alexnet()

In [ ]:
criterion = nn.CrossEntropyLoss()   # good for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.01)


In [ ]:
epochs = 10  # try 10–20 if dataset is bigger

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        # Convert 4-channel images to 3-channel by discarding the alpha channel
        images = images[:, :3, :, :]
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

In [ ]:
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")